In [ ]:
import pandas as pd

train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")

## Шаг 1. Отбор признаков

In [ ]:
feature_cols = [
    "CreditScore", "Geography", "Gender", "Age", "Tenure",
    "Balance", "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary",
]
target_col = "Exited"

X = train[feature_cols]
y = train[target_col]

## Шаг 2. Train/val split со стратификацией

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)

## Шаг 3. Pipeline: кодирование + модель

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

categorical_cols = ["Geography", "Gender"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first"), categorical_cols),
    ("num", StandardScaler(), numeric_cols),
])

model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)

## Шаг 4. Оценка на валидации: ROC AUC

In [ ]:
from sklearn.metrics import roc_auc_score

val_proba = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, val_proba)
print(f"Validation ROC AUC: {val_auc:.4f}")

## Шаг 5. Инференс на test.csv и файл сабмита

In [ ]:
test_proba = model.predict_proba(test[feature_cols])[:, 1]

submission = pd.DataFrame({
    "id": test["id"],
    "Exited": test_proba,
})
submission.to_csv("../submissions/baseline_logreg.csv", index=False)

## Шаг 8. Baseline зафиксирован

```
Baseline: LogisticRegression + OneHot(Geography, Gender) + StandardScaler
Признаки: CreditScore, Geography, Gender, Age, Tenure, Balance,
          NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary
Локальный ROC AUC (val, stratified split, random_state=42): 0.8692
Kaggle public score: 0.89026
Kaggle private score (итоговый): 0.87486
Дата: 2026-09-30
```

Локальный и приватный score близки (0.869 vs 0.875) — валидации можно доверять,
признаков утечки данных нет. До цели проекта (топ-10 ≈ 0.933+) разрыв ещё большой —
ожидаемо для модели без feature engineering. Это отправная точка для сравнения
с туториалами 4-6.